# Zadanie 1

In [5]:
module type HUFFMAN = sig
    type 'a code_tree
    type 'a code_dict
    val code_tree : 'a list -> 'a code_tree
    val dict_of_code_tree : 'a code_tree -> 'a code_dict
    val encode : 'a list -> 'a code_dict -> int list
    val decode : int list -> 'a code_tree -> 'a list
end

module type HUFFMAN =
  sig
    type 'a code_tree
    type 'a code_dict
    val code_tree : 'a list -> 'a code_tree
    val dict_of_code_tree : 'a code_tree -> 'a code_dict
    val encode : 'a list -> 'a code_dict -> int list
    val decode : int list -> 'a code_tree -> 'a list
  end


In [6]:
module type DICT = sig
    type ('a, 'b) dict
    val empty : ('a, 'b) dict
    val insert : 'a -> 'b -> ('a, 'b) dict -> ('a, 'b) dict
    val remove : 'a -> ('a, 'b) dict -> ('a, 'b) dict
    val find_opt : 'a -> ('a, 'b) dict -> 'b option
    val find : 'a -> ('a, 'b) dict -> 'b
    val to_list : ('a, 'b) dict -> ('a * 'b) list
end

module type DICT =
  sig
    type ('a, 'b) dict
    val empty : ('a, 'b) dict
    val insert : 'a -> 'b -> ('a, 'b) dict -> ('a, 'b) dict
    val remove : 'a -> ('a, 'b) dict -> ('a, 'b) dict
    val find_opt : 'a -> ('a, 'b) dict -> 'b option
    val find : 'a -> ('a, 'b) dict -> 'b
    val to_list : ('a, 'b) dict -> ('a * 'b) list
  end


In [7]:
module type PRIO_QUEUE = sig
    type ('a, 'b) pq
    
    val empty : ('a, 'b) pq
    val insert : 'a -> 'b -> ('a, 'b) pq -> ('a, 'b) pq
    val pop : ('a, 'b) pq -> ('a, 'b) pq
    val min_with_prio : ('a, 'b) pq -> 'a * 'b
end

module type PRIO_QUEUE =
  sig
    type ('a, 'b) pq
    val empty : ('a, 'b) pq
    val insert : 'a -> 'b -> ('a, 'b) pq -> ('a, 'b) pq
    val pop : ('a, 'b) pq -> ('a, 'b) pq
    val min_with_prio : ('a, 'b) pq -> 'a * 'b
  end


In [8]:
module Huffman = functor (Dict : DICT) (PrioQueue : PRIO_QUEUE) -> struct
    type 'a code_tree = CTLeaf of 'a | CTNode of 'a code_tree * 'a code_tree
    type 'a code_dict = ('a, int list) Dict.dict

    let find_else x a d =
        Option.value ~default:0 (Dict.find_opt x d)
    
    let freq_dict xs =
        let rec it xs d =
        match xs with
        | [] -> d
        | x :: xs' -> 
            it xs' (Dict.insert x 
                    (1 + find_else x 0 d) d)
        in it xs Dict.empty
    
    let initial_pq xs =
        List.fold_left (fun q (x, n) -> PrioQueue.insert n (CTLeaf x) q)
            PrioQueue.empty xs
    
    let rec algo q =
        let p1, t1 = PrioQueue.min_with_prio q
        and q1 = PrioQueue.pop q
        in if q1 = PrioQueue.empty then t1
        else let p2, t2 = PrioQueue.min_with_prio q1
        and q2 = PrioQueue.pop q1
        in algo (PrioQueue.insert (p1 + p2) (CTNode (t1, t2)) q2)
    
    let make_code_tree d = 
      Dict.to_list d |> initial_pq |> algo
    
    let code_tree xs = make_code_tree (freq_dict xs)
    
    let dict_of_code_tree t =
        let rec aux t rcpref d =
            match t with
            | CTLeaf x -> Dict.insert x (List.rev rcpref) d
            | CTNode (l, r) -> aux l (0 :: rcpref) (aux r (1 :: rcpref) d)
        in aux t [] Dict.empty
    
    let encode xs d =
        List.fold_right (@) (List.map (fun x -> Dict.find x d) xs) []
    
    let decode bs t =
        let rec walk bs cur_t =
            match cur_t with
            | CTLeaf v -> v :: start bs
            | CTNode (l, r) ->
                match bs with
                | 0 :: bs' -> walk bs' l
                | 1 :: bs' -> walk bs' r
                | _ :: _ -> failwith "a value other than 0 or 1 encountered"
                | [] -> failwith "incomplete code"
        and start bs =
            if bs = [] then [] else walk bs t
        in start bs
end

module Huffman :
  functor (Dict : DICT) (PrioQueue : PRIO_QUEUE) ->
    sig
      type 'a code_tree =
          CTLeaf of 'a
        | CTNode of 'a code_tree * 'a code_tree
      type 'a code_dict = ('a, int list) Dict.dict
      val find_else : 'a -> 'b -> ('a, int) Dict.dict -> int
      val freq_dict : 'a list -> ('a, int) Dict.dict
      val initial_pq : ('a * 'b) list -> ('b, 'a code_tree) PrioQueue.pq
      val algo : (int, 'a code_tree) PrioQueue.pq -> 'a code_tree
      val make_code_tree : ('a, int) Dict.dict -> 'a code_tree
      val code_tree : 'a list -> 'a code_tree
      val dict_of_code_tree : 'a code_tree -> ('a, int list) Dict.dict
      val encode : 'a list -> ('a, 'b list) Dict.dict -> 'b list
      val decode : int list -> 'a code_tree -> 'a list
    end


In [101]:
module ListPrioQueue : PRIO_QUEUE = struct
    type ('a, 'b) pq = ('a * 'b) list

    let empty = []
    let rec insert a x q = match q with
    | [] -> [a, x]
    | (b, y) :: ys -> if a < b then (a, x) :: q else (b, y) :: insert a x ys
    let pop q = List.tl q
    let min_with_prio q = List.hd q
end

module ListPrioQueue : PRIO_QUEUE


In [102]:
module ListDist : DICT = struct
    type ('a, 'b) dict = ('a * 'b) list
    let empty = []
    let remove k d = List.filter (fun (k', _) -> k <> k') d
    let insert k v d = (k, v) :: remove k d
    let find_opt k d = List.find_opt (fun (k', _) -> k = k') d |> Option.map snd
    let find k d = List.find(fun(k', _) -> k = k') d |> snd
    let to_list d = d
end

module ListDist : DICT


In [103]:
module MyHuffman = Huffman(ListDist) (ListPrioQueue)

module MyHuffman :
  sig
    type 'a code_tree =
      'a Huffman(ListDist)(ListPrioQueue).code_tree =
        CTLeaf of 'a
      | CTNode of 'a code_tree * 'a code_tree
    type 'a code_dict = ('a, int list) ListDist.dict
    val find_else : 'a -> 'b -> ('a, int) ListDist.dict -> int
    val freq_dict : 'a list -> ('a, int) ListDist.dict
    val initial_pq : ('a * 'b) list -> ('b, 'a code_tree) ListPrioQueue.pq
    val algo : (int, 'a code_tree) ListPrioQueue.pq -> 'a code_tree
    val make_code_tree : ('a, int) ListDist.dict -> 'a code_tree
    val dict_of_code_tree : 'a code_tree -> ('a, int list) ListDist.dict
    val encode : 'a list -> ('a, 'b list) ListDist.dict -> 'b list
    val decode : int list -> 'a code_tree -> 'a list
  end


# Zadanie 2

In [48]:
module type DICT = sig
    type key
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
end

module type DICT =
  sig
    type key
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


In [57]:
module ListDict : DICT with type key = string = struct
    type key = string
    type 'a dict = (key * 'a) list
    let empty = []
    let remove k d = List.filter(fun (k', _) -> (String.compare k k') <> 0) d
    let insert k v d = (k, v) :: remove k d
    let find_opt k d = List.find_opt(fun (k', _) -> (String.compare k k') = 0) d |> Option.map snd
    let find k d = List.find(fun(k', _) -> k = k') d |> snd
    let to_list d = d
end

module ListDict :
  sig
    type key = string
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


In [61]:
let my_dict = ListDict.empty

val my_dict : 'a ListDict.dict = <abstr>


In [62]:
let my_dict = ListDict.insert "abaab" 1 my_dict

val my_dict : int ListDict.dict = <abstr>


In [63]:
ListDict.to_list my_dict

- : (ListDict.key * int) list = [("abaab", 1)]


# Zadanie 3

In [8]:
module type DICT = sig
    type key
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
end

module type DICT =
  sig
    type key
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


In [14]:
module MakeListDict (OrderedType : Map.OrderedType) : DICT with type key = OrderedType.t = struct
    type key = OrderedType.t
    type 'a dict = (key * 'a) list
    let empty = []
    let remove k d = List.filter(fun (k', _) -> (OrderedType.compare k k') <> 0) d
    let insert k v d = (k, v) :: remove k d
    let find_opt k d = List.find_opt(fun (k', _) -> (OrderedType.compare k k') = 0) d |> Option.map snd
    let find k d = List.find(fun(k', _) -> k = k') d |> snd
    let to_list d = d
end

module MakeListDict :
  functor (OrderedType : Map.OrderedType) ->
    sig
      type key = OrderedType.t
      type 'a dict
      val empty : 'a dict
      val insert : key -> 'a -> 'a dict -> 'a dict
      val remove : key -> 'a dict -> 'a dict
      val find_opt : key -> 'a dict -> 'a option
      val find : key -> 'a dict -> 'a
      val to_list : 'a dict -> (key * 'a) list
    end


In [15]:
module M = MakeListDict (Char)

module M :
  sig
    type key = Char.t
    type 'a dict = 'a MakeListDict(Char).dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


In [18]:
let dict = M.to_list (M.insert 'c' 5 M.empty)

val dict : (M.key * int) list = [('c', 5)]


In [5]:
module CharListDict : DICT with type key = char = MakeListDict (Char)

module CharListDict :
  sig
    type key = char
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


In [7]:
let my_dict = CharListDict.empty

val my_dict : 'a CharListDict.dict = <abstr>


In [8]:
let my_dict = CharListDict.insert 'c' 5 my_dict

val my_dict : int CharListDict.dict = <abstr>


In [9]:
CharListDict.to_list my_dict

- : (CharListDict.key * int) list = [('c', 5)]


# Zadanie 4

In [5]:
module type DICT = sig
    type key
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
end

module type DICT =
  sig
    type key
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


In [2]:
module MakeMapDict = functor (OrderedType : Map.OrderedType) -> struct
    module MyMap = Map.Make(OrderedType)
    type key = OrderedType.t
    type 'a dict = 'a MyMap.t
    let empty = MyMap.empty
    let remove k d = MyMap.remove k d
    let insert k v d = MyMap.add k v d
    let find_opt k d = MyMap.find_opt k d
    let find k d = MyMap.find k d
    let to_list d = MyMap.bindings d
end

module MakeMapDict :
  functor (OrderedType : Map.OrderedType) ->
    sig
      module MyMap :
        sig
          type key = OrderedType.t
          type 'a t = 'a Map.Make(OrderedType).t
          val empty : 'a t
          val is_empty : 'a t -> bool
          val mem : key -> 'a t -> bool
          val add : key -> 'a -> 'a t -> 'a t
          val update : key -> ('a option -> 'a option) -> 'a t -> 'a t
          val singleton : key -> 'a -> 'a t
          val remove : key -> 'a t -> 'a t
          val merge :
            (key -> 'a option -> 'b option -> 'c option) ->
            'a t -> 'b t -> 'c t
          val union : (key -> 'a -> 'a -> 'a option) -> 'a t -> 'a t -> 'a t
          val compare : ('a -> 'a -> int) -> 'a t -> 'a t -> int
          val equal : ('a -> 'a -> bool) -> 'a t -> 'a t -> bool
          val iter : (key -> 'a -> unit) -> 'a t -> unit
          val fold : (key -> 'a -> 'b -> 'b) -> 'a t -> 'b -> 'b
          val for_all : (key -> 'a -> bool) -> 'a t -> b

In [7]:
module CharMapDict : DICT with type key = char = MakeMapDict (Char)

module CharMapDict :
  sig
    type key = char
    type 'a dict
    val empty : 'a dict
    val insert : key -> 'a -> 'a dict -> 'a dict
    val remove : key -> 'a dict -> 'a dict
    val find_opt : key -> 'a dict -> 'a option
    val find : key -> 'a dict -> 'a
    val to_list : 'a dict -> (key * 'a) list
  end


# Zadanie 5

In [1]:
module LeftistHeap = struct
    type ('a , 'b ) heap =
        | HLeaf
        | HNode of int * ('a , 'b ) heap * 'a * 'b * ('a , 'b ) heap

    let rank = function HLeaf -> 0 | HNode (n , _ , _ , _ , _ ) -> n
    
    let heap_ordered p = function
        | HLeaf -> true
        | HNode (_ , _ , p', _ , _ ) -> p <= p'
    
    let rec is_valid = function
        | HLeaf -> true
        | HNode (n , l , p , v , r ) ->
            rank r <= rank l
            && rank r + 1 = n
            && heap_ordered p l
            && heap_ordered p r
            && is_valid l
            && is_valid r
            
    let make_node p v l r =
        if (rank l) < (rank r) then
            HNode ((rank l) + 1, r, p, v, l)
        else
            HNode ((rank r) + 1, l, p, v, r)

    let rec heap_merge hl hr =
        match hl, hr with
        | HLeaf, _ -> hr
        | _, HLeaf -> hl
        | HNode(_, l1, p1, v1, r1), HNode(_, l2, p2, v2, r2) ->
            if p1 <= p2 then
                make_node p1 v1 l1 (heap_merge r1 hr)
            else
                make_node p2 v2 l2 (heap_merge hl r2)
end

module LeftistHeap :
  sig
    type ('a, 'b) heap =
        HLeaf
      | HNode of int * ('a, 'b) heap * 'a * 'b * ('a, 'b) heap
    val rank : ('a, 'b) heap -> int
    val heap_ordered : 'a -> ('a, 'b) heap -> bool
    val is_valid : ('a, 'b) heap -> bool
    val make_node :
      'a -> 'b -> ('a, 'b) heap -> ('a, 'b) heap -> ('a, 'b) heap
    val heap_merge : ('a, 'b) heap -> ('a, 'b) heap -> ('a, 'b) heap
  end


# Zadanie 6

In [2]:
module type PRIO_QUEUE = sig
  type ('a, 'b) pq
  
  val empty : ('a, 'b) pq
  val insert : 'a -> 'b -> ('a, 'b) pq -> ('a, 'b) pq
  val pop : ('a, 'b) pq -> ('a, 'b) pq
  val min_with_prio : ('a, 'b) pq -> 'a * 'b
end

module type PRIO_QUEUE =
  sig
    type ('a, 'b) pq
    val empty : ('a, 'b) pq
    val insert : 'a -> 'b -> ('a, 'b) pq -> ('a, 'b) pq
    val pop : ('a, 'b) pq -> ('a, 'b) pq
    val min_with_prio : ('a, 'b) pq -> 'a * 'b
  end


In [3]:
module PrioQueueHeap : PRIO_QUEUE = struct
    open LeftistHeap (* using namespace LeftistHeap *)
    type ('a, 'b) pq  = ('a, 'b) heap
    let empty = HLeaf
    let insert a x q = heap_merge (make_node a x HLeaf HLeaf) q
    let pop q = match q with
        | HLeaf -> HLeaf
        | HNode(_, l, p, v, r) -> heap_merge l r
    let min_with_prio q = match q with
        | HLeaf -> failwith "Empty queue"
        | HNode(_, _, p, v, _) -> (p, v)
end

module PrioQueueHeap : PRIO_QUEUE


In [4]:
module MyPrioQueueSort = functor (PrioQueue : PRIO_QUEUE) -> struct
    open PrioQueue
    let rec build_pq xs = match xs with
    | [] -> empty
    | x :: xs -> insert x x (build_pq xs)
    
    let rec to_list q =
        if q = empty then []
        else fst (min_with_prio q) :: to_list (pop q)

    let pqsort xs = to_list (build_pq xs)
end

module Sort = MyPrioQueueSort(PrioQueueHeap);;

module MyPrioQueueSort :
  functor (PrioQueue : PRIO_QUEUE) ->
    sig
      val build_pq : 'a list -> ('a, 'a) PrioQueue.pq
      val to_list : ('a, 'b) PrioQueue.pq -> 'a list
      val pqsort : 'a list -> 'a list
    end


module Sort :
  sig
    val build_pq : 'a list -> ('a, 'a) PrioQueueHeap.pq
    val to_list : ('a, 'b) PrioQueueHeap.pq -> 'a list
    val pqsort : 'a list -> 'a list
  end


In [5]:
Sort.pqsort [2; 1; 3; 7; 4; 2]

- : int list = [1; 2; 2; 3; 4; 7]
